# Weather Agent Workflow for Italian Cities

**Exercise objective:** build a deterministic, multi-step workflow for an agent that answers weather questions about the largest Italian cities. The agent is not free-form: each step has a fixed responsibility, and the LLM is invoked only where natural language understanding or generation is strictly necessary.

**Pipeline overview:**

1. **Extract** the city and request type from the user's natural-language question.
2. **Validate** that the city is supported and the request is meaningful.
3. **Dispatch** the correct tool (or both tools) based on the request type.
4. **Generate** a short natural-language answer from the tool outputs.

**Tools to implement (static data, no live API):**

| Tool | Signature | Purpose |
|------|-----------|---------|
| `get_temperature` | `(city: str) -> float` | Current temperature in degrees Celsius |
| `get_overall_weather_conditions` | `(city: str) -> str` | Short qualitative description (e.g. `"sunny"`, `"heavy rain"`) |

**Supported cities:** Roma, Milano, Napoli, Torino, Firenze, Bologna, Venezia, Palermo, Bari, Genova.

**Design choices worth flagging upfront:**

- The workflow is **rule-based at the orchestration level** (Python `if`/`elif` decides which tool to call), and **LLM-based only at the boundaries** (parsing the user input, phrasing the final answer). This is a classic *workflow agent* in Anthropic's terminology, sitting one notch below a fully autonomous agent: the LLM does not decide the control flow, which makes the system far easier to debug and to test.
- The tools return **mock data** because the goal of the exercise is the orchestration logic, not the integration with a real weather API. Swapping in a real API (e.g. Open-Meteo) would only touch the body of the two tool functions, not the surrounding workflow.

**Stack:** `litellm` against a local [Ollama](https://ollama.com) server (model: `llama3.2`). No API key required; the model runs entirely on the local machine. Switching to a hosted provider later is a one-line change of the `MODEL` constant.

> **Prerequisite:** the Ollama daemon must be running (`ollama serve`) and the model already pulled (`ollama pull llama3.2`). Both checks are enforced in the Setup cell.

---
## 1. Setup

Two utilities will be reused across every workflow step:

- `call_llm(messages)` wraps `litellm.completion` so the rest of the notebook never touches the SDK directly. Centralising the call also makes it trivial to swap model, temperature, or provider later without hunting through cells.
- `parse_json(text)` extracts the first JSON object from a raw model response. LLMs frequently wrap structured outputs in prose (`"Here's the JSON: { ... }"`), so a regex-based extractor is more robust than `json.loads` on the raw text. The regex is non-greedy and matches a single flat object, which is exactly what every step in this workflow produces.

A note about the choice of `llama3.2` (3B) as the backing model: it is less consistent than larger hosted models (e.g. `gpt-4o-mini`) at returning *strictly* valid JSON with no surrounding prose. The combination of a tightly-constrained system prompt, low temperature, and the regex-based `parse_json` is enough to absorb that variance for the structured-extraction step.

In [15]:
# Install once if needed:
# %pip install litellm

import json
import re
import subprocess

from litellm import completion

# Sanity-check that the local Ollama server has the model we need.
# Failing here is cheaper and louder than failing inside the first LLM call,
# and the error message tells the student exactly how to recover.
_ollama_list = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(_ollama_list.stdout)
assert "llama3.2" in _ollama_list.stdout, (
    "llama3.2 not found - start the daemon with `ollama serve` and run `ollama pull llama3.2`."
)

# litellm routes `ollama_chat/<model>` to the local Ollama chat-completion endpoint.
# Swapping provider later is a one-line change (e.g. "groq/llama-3.1-8b-instant"
# or "openai/gpt-4o-mini" + the matching API key in the environment).
MODEL = "ollama_chat/llama3.2"

NAME                       ID              SIZE      MODIFIED   
llama3.2:latest            a80c4f17acd5    2.0 GB    4 days ago    
glm-ocr:latest             6effedd0dc8a    2.2 GB    4 days ago    
qwen2.5:14b                7cdf5a0187d5    9.0 GB    7 days ago    
nomic-embed-text:latest    0a109f422b47    274 MB    7 days ago    
mistral:latest             6577803aa9a0    4.4 GB    7 days ago    



In [16]:
# ── Shared utilities ───────────────────────────────────────────────────────────

def call_llm(messages: list[dict], temperature: float = 0.2) -> str:
    """Send a chat-style request to the configured model and return the text content.

    Default temperature is low: extraction and short answers benefit from determinism,
    and any step that needs more creativity can override it.
    """
    response = completion(model=MODEL, messages=messages, temperature=temperature)
    return response.choices[0].message.content


def parse_json(text: str) -> dict | None:
    """Extract the first flat JSON object found in a string, or None if absent.

    The regex is non-greedy and does not handle nested objects on purpose: every
    structured output produced in this workflow is a single flat dict, and refusing
    to parse nested JSON makes failure modes explicit rather than silently wrong.
    """
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None


# Smoke test: round-trip a trivial JSON request through the model.
_probe = call_llm([
    {"role": "system", "content": "Reply with valid JSON only."},
    {"role": "user", "content": 'Return {"ok": true}.'},
])
print("Raw model output:", _probe)
print("Parsed         :", parse_json(_probe))

Raw model output: {"ok":true}
Parsed         : {'ok': True}


---
## 2. Agent Tools

Two tools, both backed by an in-memory dictionary keyed by city name. A few notes on the design:

- **Static data instead of a live API.** The point of the exercise is the orchestration, not the network plumbing. Hardcoding the dataset keeps the notebook reproducible (no rate limits, no flaky DNS) and makes the workflow's behaviour deterministic, which is essential when debugging the LLM-driven steps later. A real implementation would swap the dictionary lookup for an HTTP call to a provider like Open-Meteo - the function signature would not change.
- **Strict input validation.** Each tool raises `ValueError` if the city is not in the dataset. The validation step in the workflow will catch unsupported cities *before* the tool is ever called, so this guard is a defence-in-depth measure: if a future caller skips validation, the tool still refuses to fabricate an answer.
- **Conditions in English.** The agent ultimately produces an English-language answer, so the conditions strings are kept in English directly. This avoids an extra translation step inside the answer-generation prompt and keeps the data source aligned with the output language.
- **Single source of truth for supported cities.** `SUPPORTED_CITIES` is derived from `TEMPERATURE_DATA`. The two dictionaries must share the same keys; deriving the list rather than maintaining a separate constant prevents drift.

In [17]:
# ── Mock weather database ──────────────────────────────────────────────────────
# Values are realistic for an Italian winter afternoon and were chosen so that the
# answer-generation step has clearly distinguishable cases (a sunny mild city, a
# snowy cold one, a foggy one, etc.) when we test it later.

TEMPERATURE_DATA: dict[str, float] = {
    "Roma":    14.0,
    "Milano":  10.0,
    "Napoli":  16.0,
    "Torino":   8.0,
    "Firenze": 12.0,
    "Bologna": 11.0,
    "Venezia":  9.0,
    "Palermo": 18.0,
    "Bari":    15.0,
    "Genova": 13.0,
}

WEATHER_CONDITIONS_DATA: dict[str, str] = {
    "Roma":    "sunny",
    "Milano":  "rain with sunny spells",
    "Napoli":  "cloudy",
    "Torino":  "light snow",
    "Firenze": "partly cloudy",
    "Bologna": "fog",
    "Venezia": "high tide and rain",
    "Palermo": "sunny and windy",
    "Bari":    "cloudy with wind",
    "Genova": "heavy rain",
}

SUPPORTED_CITIES: list[str] = list(TEMPERATURE_DATA.keys())


def get_temperature(city: str) -> float:
    """Return the current temperature in degrees Celsius for a supported city."""
    if city not in TEMPERATURE_DATA:
        raise ValueError(
            f"City '{city}' is not supported. Available cities: {', '.join(SUPPORTED_CITIES)}"
        )
    return TEMPERATURE_DATA[city]


def get_overall_weather_conditions(city: str) -> str:
    """Return a short qualitative description of the current weather for a supported city."""
    if city not in WEATHER_CONDITIONS_DATA:
        raise ValueError(
            f"City '{city}' is not supported. Available cities: {', '.join(SUPPORTED_CITIES)}"
        )
    return WEATHER_CONDITIONS_DATA[city]

In [18]:
# ── Smoke tests for the tools ──────────────────────────────────────────────────
# Cover the happy path on two cities and the failure path on an unsupported one.
# These checks run in milliseconds and catch typos in the dataset early.

print("get_temperature:")
for city in ("Roma", "Milano"):
    print(f"  {city:8s} -> {get_temperature(city):.1f} °C")

print("\nget_overall_weather_conditions:")
for city in ("Roma", "Milano"):
    print(f"  {city:8s} -> {get_overall_weather_conditions(city)}")

print("\nUnsupported city:")
try:
    get_temperature("London")
except ValueError as exc:
    print(f"  ValueError correctly raised: {exc}")

get_temperature:
  Roma     -> 14.0 °C
  Milano   -> 10.0 °C

get_overall_weather_conditions:
  Roma     -> sunny
  Milano   -> rain with sunny spells

Unsupported city:
  ValueError correctly raised: City 'London' is not supported. Available cities: Roma, Milano, Napoli, Torino, Firenze, Bologna, Venezia, Palermo, Bari, Genova


---
## 3. Workflow Steps

Each subsection introduces a single stage: rationale first, function next, focused test cell at the end. The end-to-end orchestrator is assembled in Section 4 once all four stages exist and have been validated in isolation.

### 3.1 - Extraction step

The first stage maps a free-form user question to a structured payload:

```json
{"city": "<Italian city name, capitalised, or null>", "request_type": "temperature | conditions | both | null"}
```

**Prompt design choices:**

- **Few-shot examples over verbal description.** llama3.2 (3B) is not as reliable as larger hosted models at following abstract formatting instructions. Concrete input/output pairs anchor the model's behaviour far more effectively. Four examples cover the four valid `request_type` values, which is the minimum that demonstrates each branch without bloating context.
- **The extractor lets unsupported cities through.** If the user asks about *London*, the model is allowed to return `{"city": "London", ...}` and the next step rejects it. Filtering here would conflate two responsibilities (extraction vs domain validation) and force every system prompt to carry the supported-cities list, inflating tokens with no benefit.
- **`null` is a first-class value.** Both `city` and `request_type` can be `null`. Modelling absence as `null` (instead of raising) keeps the failure path explicit and uniform: the validation step is the single place that turns "missing information" into a user-facing error message.
- **Temperature 0.0 for this call.** The task is deterministic mapping; creativity hurts. The lower the temperature, the lower the chance of the model surrounding the JSON with chatty prose.
- **Italian capitalisation enforced in the prompt, not in Python.** The dataset keys are `"Roma"`, `"Milano"`, etc. Instructing the model to normalise capitalisation at extraction time avoids a post-processing step and keeps the contract between extraction and validation simple (`city in SUPPORTED_CITIES` is a plain string check).

In [19]:
# ── Step 1: extract (city, request_type) from a natural-language question ─────

EXTRACTION_SYSTEM_PROMPT = """You analyse the user's weather question and extract two fields as strict JSON.

Reply with a single JSON object and nothing else. No prose, no markdown fences.

Schema:
{
  "city": "<Italian city name with the first letter capitalised, or null if not mentioned>",
  "request_type": "temperature" | "conditions" | "both" | null
}

Rules for request_type:
- "temperature" -> the user asks how many degrees, the current temperature, how hot or cold it is.
- "conditions"  -> the user asks what the weather is like, the forecast, the atmospheric conditions.
- "both"        -> the user asks generic weather information or both temperature and conditions explicitly.
- null          -> the question has nothing to do with the weather.

Examples:
User: How many degrees is it in Milan?
Output: {"city": "Milano", "request_type": "temperature"}

User: What's the weather like in Naples today?
Output: {"city": "Napoli", "request_type": "conditions"}

User: Tell me everything about the weather in Rome
Output: {"city": "Roma", "request_type": "both"}

User: What are you doing tonight?
Output: {"city": null, "request_type": null}
"""


def extraction_step(question: str) -> dict:
    """Turn a free-form user question into a structured {city, request_type} dict.

    Raises ValueError if the model output does not contain a parsable JSON object;
    that signals a prompt problem and should fail loudly rather than silently
    feeding garbage into the rest of the workflow.
    """
    raw = call_llm(
        messages=[
            {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature=0.0,  # deterministic structured extraction
    )

    parsed = parse_json(raw)
    if parsed is None:
        raise ValueError(f"Extraction step could not parse JSON from model output: {raw!r}")

    print(f"   [1] extraction -> {json.dumps(parsed, ensure_ascii=False)}")
    return parsed

In [20]:
# ── Test the extraction step in isolation ─────────────────────────────────────
# The four questions exercise the four valid combinations of (city present?,
# request_type). The last two are deliberately tricky: a non-weather question
# (city must be null) and an unsupported city (extraction succeeds, validation
# will catch it later). Running the step in isolation keeps debugging cheap -
# any failure here is a prompt problem, not an orchestration bug.

test_questions = [
    "How many degrees is it in Milan?",          # temperature, Milano
    "What's the weather like in Naples today?",  # conditions, Napoli
    "Tell me everything about the weather in Rome",  # both, Roma
    "What's 1+1?",                                # off-domain, expect nulls
    "What's the weather in London?",              # unsupported city, expect London
]

print("Extraction step - per-question outputs:\n")
for q in test_questions:
    print(f"Q: {q}")
    try:
        extracted = extraction_step(q)
    except ValueError as exc:
        print(f"   error: {exc}\n")
        continue
    print()

Extraction step - per-question outputs:

Q: How many degrees is it in Milan?
   [1] extraction -> {"city": "Milano", "request_type": "temperature"}

Q: What's the weather like in Naples today?
   [1] extraction -> {"city": "Napoli", "request_type": "conditions"}

Q: Tell me everything about the weather in Rome
   [1] extraction -> {"city": "Roma", "request_type": "both"}

Q: What's 1+1?
   [1] extraction -> {"city": null, "request_type": null}

Q: What's the weather in London?
   [1] extraction -> {"city": "London", "request_type": "conditions"}



### 3.2 - Validation step

Pure-Python check on the structured output of step 3.1. No LLM call - validation is deterministic logic and routing it through a model would be both wasteful and unreliable.

**What it checks, in order of cheapness:**

1. `city` is not null (the user mentioned a city at all).
2. `city` is in `SUPPORTED_CITIES` (we have data for it).
3. `request_type` is not null (we know what to do with it).

**Return contract:** `(True, "")` on success, `(False, <user-facing error message>)` on failure. Returning the message alongside the boolean lets the orchestrator forward it directly to the user without crafting a new one - the validator is the single source of truth for negative responses, which keeps them consistent across calls.

**Why not raise exceptions?** Two reasons:

- Validation failures are *expected*, not exceptional - "I don't recognise that city" is a normal part of the conversation. Exceptions are reserved for genuine programming errors (e.g. the LLM in step 3.1 returning unparsable JSON).
- A tuple return lets the orchestrator short-circuit with a regular `if` instead of `try/except` machinery, which keeps the main flow readable.

In [21]:
# ── Step 2: validate the extracted payload before calling any tool ────────────

def validation_step(extracted: dict) -> tuple[bool, str]:
    """Validate the (city, request_type) payload from step 3.1.

    Returns:
        (True, "")           if every field is acceptable.
        (False, <message>)   where <message> is suitable for direct display to the user.
    """
    city = extracted.get("city")
    request_type = extracted.get("request_type")

    if not city:
        msg = "I could not work out which city you mean. Could you tell me explicitly?"
        print("   [2] validation -> error (no city)")
        return False, msg

    if city not in SUPPORTED_CITIES:
        msg = (
            f"Sorry, '{city}' is not one of the cities I cover. "
            f"Available cities: {', '.join(SUPPORTED_CITIES)}."
        )
        print(f"   [2] validation -> error (unsupported city: {city})")
        return False, msg

    if not request_type:
        msg = "I couldn't tell what you want to know about the weather. Try asking for the temperature or the conditions."
        print("   [2] validation -> error (no request_type)")
        return False, msg

    print("   [2] validation -> ok")
    return True, ""

In [22]:
# ── Test the validation step ──────────────────────────────────────────────────
# Five cases, covering every accept and reject path:
#   1. happy path with a city we cover
#   2. happy path with request_type='both'
#   3. reject: extraction failed to find a city
#   4. reject: city extracted but not in our dataset
#   5. reject: extraction failed to find a request_type

validation_cases = [
    {"city": "Milano",  "request_type": "temperature"},
    {"city": "Firenze", "request_type": "both"},
    {"city": None,      "request_type": "conditions"},
    {"city": "London",  "request_type": "temperature"},
    {"city": "Roma",    "request_type": None},
]

for case in validation_cases:
    ok, msg = validation_step(case)
    label = "OK   " if ok else "FAIL "
    print(f"{label} {case}")
    if not ok:
        print(f"       message: {msg}")
    print()

   [2] validation -> ok
OK    {'city': 'Milano', 'request_type': 'temperature'}

   [2] validation -> ok
OK    {'city': 'Firenze', 'request_type': 'both'}

   [2] validation -> error (no city)
FAIL  {'city': None, 'request_type': 'conditions'}
       message: I could not work out which city you mean. Could you tell me explicitly?

   [2] validation -> error (unsupported city: London)
FAIL  {'city': 'London', 'request_type': 'temperature'}
       message: Sorry, 'London' is not one of the cities I cover. Available cities: Roma, Milano, Napoli, Torino, Firenze, Bologna, Venezia, Palermo, Bari, Genova.

   [2] validation -> error (no request_type)
FAIL  {'city': 'Roma', 'request_type': None}
       message: I couldn't tell what you want to know about the weather. Try asking for the temperature or the conditions.



### 3.3 - Tool dispatch step

Once the extracted payload has passed validation, this step calls the right tool(s) based on `request_type`. Like the validator, this is pure Python: a tight `if` ladder, no LLM in the loop.

**Why an explicit dispatch instead of letting the model choose the tool?**

- **Determinism.** With three valid `request_type` values, the mapping to tool calls is unambiguous. A model in the loop could decide differently on the same input on a different day. For a workflow where the dispatch criteria are this clear, that variability has no upside.
- **Observability.** Explicit dispatch makes the chosen tool visible in the trace, which simplifies debugging and post-hoc analysis. Function-calling APIs (OpenAI tool use, Anthropic tool use, etc.) are the right choice when the set of tools is large or when dispatch criteria are fuzzy; here it would be over-engineering.
- **Cost.** A tool-calling LLM round-trip would add one model call per question with no behavioural gain.

**Return shape:** a dict with both keys, set to `None` for the tool that was not called. Keeping both keys present at all times means the answer-generation step in 3.4 can iterate over the same structure without conditionals.

In [23]:
# ── Step 3: dispatch the right tool(s) based on the request_type ──────────────

def tool_dispatch_step(city: str, request_type: str) -> dict:
    """Call the temperature tool, the conditions tool, or both.

    Trusts that 'city' and 'request_type' have already been validated; calling it
    with bad input is a programming error and the underlying tools will raise
    ValueError as a last line of defence.

    Returns a dict {'temperature': float|None, 'conditions': str|None}.
    """
    temperature = None
    conditions = None

    if request_type in ("temperature", "both"):
        temperature = get_temperature(city)
    if request_type in ("conditions", "both"):
        conditions = get_overall_weather_conditions(city)

    parts = []
    if temperature is not None:
        parts.append(f"temperature={temperature} °C")
    if conditions is not None:
        parts.append(f"conditions='{conditions}'")
    print(f"   [3] tool dispatch ({request_type}) -> {', '.join(parts)}")

    return {"temperature": temperature, "conditions": conditions}

In [24]:
# ── Test the tool dispatch step ───────────────────────────────────────────────
# Cover all three branches with three different cities so the underlying data also
# varies (a sunny city, a rainy one, a cloudy one).

print(tool_dispatch_step("Roma",   "temperature"))
print(tool_dispatch_step("Milano", "conditions"))
print(tool_dispatch_step("Napoli", "both"))

   [3] tool dispatch (temperature) -> temperature=14.0 °C
{'temperature': 14.0, 'conditions': None}
   [3] tool dispatch (conditions) -> conditions='rain with sunny spells'
{'temperature': None, 'conditions': 'rain with sunny spells'}
   [3] tool dispatch (both) -> temperature=16.0 °C, conditions='cloudy'
{'temperature': 16.0, 'conditions': 'cloudy'}


### 3.4 - Answer generation step

The last stage converts tool outputs back into a natural-language sentence. The LLM returns to the loop, but its responsibility is narrow: phrase the answer, do not invent or recompute facts.

**Prompt design choices:**

- **Data first, instruction next.** Tool outputs are inlined in the user message rather than the system prompt, so the system prompt stays stable across calls and the user message carries the per-question payload. Same pattern used by LangChain's chat-with-context templates.
- **Length cap (1 to 2 sentences, hard ceiling of 4).** Without an explicit constraint, a 3B model tends to over-explain. The cap also makes responses comparable across cities in the demos.
- **Warm but concise tone.** Mirrors a friendly chatbot register without the "Hello! It's lovely to chat with you today" register that smaller models default to.
- **Temperature 0.5.** Higher than extraction (we want a bit of variety in phrasing) but well below 1.0 (we don't want fabricated facts or sudden tonal swings).
- **No mention of missing fields.** Only the values the tool actually produced are passed in the prompt, so the model has nothing to hallucinate from. If the request was `temperature` only, the user message will contain a single key and the answer will mention temperature only.

In [25]:
# ── Step 4: generate a short natural-language answer ──────────────────────────

ANSWER_SYSTEM_PROMPT = """You are a friendly weather assistant. You receive a city name and the tool outputs for that city.

Write a short answer in English that uses only the data provided.
Hard constraints:
- 1 to 2 sentences, never more than 4.
- Warm but concise tone, no greetings, no filler.
- Do not invent any data that is not in the input.
- If only the temperature is given, talk about temperature only. If only conditions, talk about conditions only.
"""


def answer_generation_step(city: str, tool_results: dict) -> str:
    """Render the tool outputs as a short natural-language sentence."""
    info_parts = []
    if tool_results.get("temperature") is not None:
        info_parts.append(f"temperature: {tool_results['temperature']} °C")
    if tool_results.get("conditions") is not None:
        info_parts.append(f"conditions: {tool_results['conditions']}")

    user_message = f"City: {city}\nData: {', '.join(info_parts)}"

    answer = call_llm(
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.5,  # mild variety, no hallucination
    ).strip()

    print(f"   [4] answer -> generated ({len(answer.split())} words)")
    return answer

In [26]:
# ── Test the answer generation step ───────────────────────────────────────────
# Cover the three input shapes (temperature only, conditions only, both) plus a
# severe-weather case to check the model stays factual (does not dramatise
# 'high tide and rain' into a flood warning).

cases = [
    ("Roma",    {"temperature": 14.0, "conditions": None}),
    ("Milano",  {"temperature": None, "conditions": "rain with sunny spells"}),
    ("Napoli",  {"temperature": 16.0, "conditions": "cloudy"}),
    ("Venezia", {"temperature":  9.0, "conditions": "high tide and rain"}),
]

for city, results in cases:
    answer = answer_generation_step(city, results)
    print(f"\n{city}: {answer}\n")

   [4] answer -> generated (22 words)

Roma: Rome's current temperature is a pleasant 14 degrees Celsius, ideal for exploring the city's ancient ruins and indulging in delicious Italian cuisine.

   [4] answer -> generated (25 words)

Milano: Milano can be quite unpredictable, with rain showers often giving way to sunny spells. This mixed weather pattern is typical of the city's Mediterranean climate.

   [4] answer -> generated (38 words)

Napoli: Napoli today is experiencing a mild and overcast day with a temperature of 16.0 °C. The cloudy skies add to the city's relaxed atmosphere, making it an ideal day for exploring its historic sites and enjoying local cuisine.

   [4] answer -> generated (35 words)

Venezia: Venezia experiences a cool day with a temperature of 9.0°C, making it an ideal time to explore the city's indoor attractions. However, the presence of both high tide and rain will make outdoor activities challenging.



---
## 4. Orchestrator: end-to-end workflow

The orchestrator chains the four stages in order. It is a single Python function: no LLM in the loop, no recursion, no retries. The output is always a dict with the same shape, regardless of whether the call succeeded or hit an error, so downstream consumers (a CLI, a web handler) can treat both cases uniformly.

**Return shape:**

```python
{
    "state":        "success" | "error",
    "city":         str | None,   # city extracted, even if validation later rejected it
    "tool_results": dict | None,  # None if we never reached the tool dispatch
    "message":      str,          # natural-language reply (success) or error message (failure)
}
```

**Error handling philosophy:**

- `extraction_step` raises `ValueError` only if the LLM returns unparsable output (a real prompt / model problem). The orchestrator catches it and reports a graceful error.
- Validation failures (no city, unsupported city, no request type) are *not* exceptions - they are a normal outcome of conversation. The orchestrator forwards the validator's message verbatim, with no extra rephrasing.
- Tool calls do not need a try/except in the orchestrator: validation guarantees `city in SUPPORTED_CITIES`, so the tools can only raise if there is a bug upstream - in which case we *want* a stack trace, not a swallowed error.

In [27]:
# ── Orchestrator: chain the four steps ────────────────────────────────────────

def workflow(question: str) -> dict:
    """Run the full extract -> validate -> dispatch -> answer pipeline for one question.

    Single entry point for the agent; downstream callers should not need to know
    about the individual steps.
    """
    print(f"User: {question}")

    # Step 1: extract
    try:
        extracted = extraction_step(question)
    except ValueError:
        return {
            "state":        "error",
            "city":         None,
            "tool_results": None,
            "message":      "I had trouble parsing your question. Could you rephrase it?",
        }

    # Step 2: validate
    is_valid, error_msg = validation_step(extracted)
    if not is_valid:
        return {
            "state":        "error",
            "city":         extracted.get("city"),
            "tool_results": None,
            "message":      error_msg,
        }

    city         = extracted["city"]
    request_type = extracted["request_type"]

    # Step 3: dispatch the tools (validation guarantees this is safe).
    tool_results = tool_dispatch_step(city, request_type)

    # Step 4: phrase the natural-language answer.
    message = answer_generation_step(city, tool_results)

    return {
        "state":        "success",
        "city":         city,
        "tool_results": tool_results,
        "message":      message,
    }

In [28]:
# ── End-to-end demos ──────────────────────────────────────────────────────────
# Five curated questions covering:
#   - every happy path (temperature, conditions, both)
#   - one unsupported-city failure (London)
#   - one off-domain failure (a distance question, not weather)

demo_questions = [
    "How many degrees is it in Milan?",
    "What's the weather like in Naples?",
    "Tell me everything about the weather in Rome",
    "What's the weather in London?",
    "How far is Milan from Genoa?",
]

all_results = []
for q in demo_questions:
    print("=" * 70)
    result = workflow(q)
    icon = "OK  " if result["state"] == "success" else "FAIL"
    print(f"{icon} state={result['state']}")
    print(f"Agent: {result['message']}")
    print()
    all_results.append(result)

User: How many degrees is it in Milan?
   [1] extraction -> {"city": "Milano", "request_type": "temperature"}
   [2] validation -> ok
   [3] tool dispatch (temperature) -> temperature=10.0 °C
   [4] answer -> generated (31 words)
OK   state=success
Agent: Milano's current temperature is quite cool at 10.0 °C. This chilly weather may be a good excuse to grab a warm cup of coffee or hot chocolate in this Italian city.

User: What's the weather like in Naples?
   [1] extraction -> {"city": "Napoli", "request_type": "conditions"}
   [2] validation -> ok
   [3] tool dispatch (conditions) -> conditions='cloudy'
   [4] answer -> generated (27 words)
OK   state=success
Agent: Napoli is currently experiencing a cloudy day with unclear skies, making it a good idea to carry an umbrella or raincoat if you plan to be outside.

User: Tell me everything about the weather in Rome
   [1] extraction -> {"city": "Roma", "request_type": "both"}
   [2] validation -> ok
   [3] tool dispatch (both) -> temper

---
## 5. Critical analysis

### What the workflow gets right

**Separation of concerns.** Each step has exactly one responsibility: extraction is the only place that calls the LLM for understanding, validation is the only place that says no, dispatch is the only place that decides which tool runs, generation is the only place that phrases an answer. Debugging is then a matter of pointing at the failing step and looking at its output in isolation - which is exactly what the per-stage test cells let us do.

**LLM at the boundaries, Python in the middle.** The LLM is used where it genuinely adds value (parsing free-form input, phrasing free-form output) and replaced by deterministic code where it does not (validation, tool dispatch). The result is a workflow that is cheaper than a fully autonomous agent (one LLM call per stage, no planning loop), more debuggable (every decision is visible in plain Python), and equally capable for this problem class.

**Defence in depth on city validation.** The validator rejects unsupported cities, but the tools themselves also raise on unknown keys. If a future refactor accidentally skipped validation, the tools would still refuse to fabricate data. The extra `if city not in TEMPERATURE_DATA: raise` line costs nothing and removes an entire class of silent-failure bugs.

**Uniform return shape from the orchestrator.** `state`, `city`, `tool_results`, `message` are always present, regardless of success or failure. A future web handler can render the response with one template instead of branching on the type of failure.

### Where llama3.2 (3B) is the weak link

The structured-extraction step is the most fragile component. A 3B model occasionally:

- Wraps the JSON in prose (`"Here is the result: { ... }"`).
- Emits `"null"` (string) instead of `null` (JSON literal).
- Normalises city names inconsistently (`"Roma"` vs `"Rome"`).
- Picks the wrong `request_type` when the question is ambiguous (e.g. *"is it cold in Milan?"* sometimes routes to `"conditions"` instead of `"temperature"`).

Three mitigations are already in place:

1. `parse_json` tolerates surrounding prose (regex extraction rather than `json.loads` on the raw string).
2. Few-shot examples in the system prompt pin every valid output shape.
3. Temperature 0.0 reduces creative deviations.

If the extraction success rate were still too low for production use, the next moves would be, in order of effort:

- Switch the local model to `llama3.1:8b` or `mistral:7b` (similar latency on an M-series Mac, noticeably better instruction-following).
- Move to a hosted model with native JSON mode (OpenAI's `response_format={"type": "json_object"}` or Gemini's `response_mime_type="application/json"`).
- Use a tool-calling API where the schema is enforced server-side and unparsable outputs are not even returned.

### Where this falls short of a true *agent*

By Anthropic's taxonomy this is a *workflow*, not an *agent*: the control flow is hardcoded by the human, not chosen by the model. The system cannot:

- Decide that two cities need to be queried in parallel (*"compare the weather in Rome and Milan"*).
- Chain reasoning across multiple turns (*"yesterday you said it was raining; is it still?"*).
- Recover from a failure by replanning (*"the temperature tool is down, try conditions and tell the user temperature is unavailable"*).

Each of these capabilities requires either a richer workflow (multiple sequential calls, conversational memory) or a step up to a true agent loop with model-driven tool selection. The cost is more LLM calls per question, less determinism, and harder debugging. For a fixed-scope FAQ bot like this one, the tradeoffs are not worth it.

### Production-readiness notes

- **Real data.** Replacing the two static dictionaries with calls to Open-Meteo (no auth needed) would not change anything outside the tool functions. Add async + timeout if multiple tools must run in parallel; for two synchronous lookups it is unnecessary.
- **Caching.** Weather data does not change minute-by-minute. A TTL cache on the tool outputs (e.g. `cachetools.TTLCache`, refresh every 10 minutes) would cut latency and external traffic for popular cities, at the cost of one extra dependency.
- **Observability.** The compact `[1] / [2] / [3] / [4]` banners are enough for notebook work but would need to become structured log lines (JSON, one per step, with the user question and each step's payload) in production. Anything that goes through an LLM should have its raw input/output retained long enough to debug a failed extraction post-hoc.
- **Cost guardrails.** Even local inference has a latency budget; for a hosted model a per-user rate limit and a max-tokens cap would be standard. The four-step layout makes guardrails easy to install: each call has a known maximum input and output size.
- **Multilingual input.** Users will mix Italian and English ("Che tempo fa a Milano?" and "What's the weather in Milan?" both make sense for these cities). The current prompt accepts both informally but does not advertise the capability. A test set that explicitly exercises both languages would be the first step before claiming bilingual support.

### When to choose this design over alternatives

| Design | When it's the right call |
|--------|--------------------------|
| **This workflow (rule-based dispatch)** | Tools are small and known, dispatch logic is deterministic, debuggability matters, cost per call is sensitive. |
| **Single-prompt agent (tool-calling API)** | Tools are many and dispatch is fuzzy, the schema is enforced by the model server, cost per call is tolerable. |
| **ReAct / multi-step agent loop** | The user's question may require chaining several tool calls with intermediate reasoning, recovering from failures, or replanning. |

For an FAQ bot scoped to ten cities and two tools, the workflow design is the cheapest, most predictable, and most debuggable of the three. Anything more elaborate would be solving a problem this exercise does not have.